In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/Cerebrasensecloud')
OASIS2_BUNDLE_ROOT = DRIVE_ROOT / 'OASIS-2'
RUNTIME_ROOT = DRIVE_ROOT / 'backend_runtime'
RUN_NAME = 'oasis2_colab_baseline'

for name, path in {
    'DRIVE_ROOT': DRIVE_ROOT,
    'OASIS2_BUNDLE_ROOT': OASIS2_BUNDLE_ROOT,
    'RUNTIME_ROOT': RUNTIME_ROOT,
}.items():
    print(name, path.exists(), path)

if not OASIS2_BUNDLE_ROOT.exists():
    raise FileNotFoundError(f'Missing uploaded OASIS-2 bundle: {OASIS2_BUNDLE_ROOT}')


In [ ]:
import shutil
import subprocess
import sys
from pathlib import Path

REPO_ROOT = Path('/content/Cerebrasense-')
BACKEND_ROOT = REPO_ROOT / 'alz_backend'

if REPO_ROOT.exists():
    shutil.rmtree(REPO_ROOT)

subprocess.run(
    ['git', 'clone', 'https://github.com/Billrichard209/Cerebrasense-.git'],
    cwd='/content',
    check=True,
)

if not BACKEND_ROOT.exists():
    raise FileNotFoundError(f'Expected backend after clone: {BACKEND_ROOT}')

requirements = BACKEND_ROOT / 'requirements-colab.txt'
subprocess.run([sys.executable, '-m', 'pip', 'install', '--upgrade', 'pip'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(requirements)], check=True)

repo_commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_ROOT, text=True).strip()
print('repo_root =', REPO_ROOT)
print('backend_root =', BACKEND_ROOT)
print('repo_commit =', repo_commit)


In [ ]:
import subprocess
import sys

cmd = [
    sys.executable,
    'scripts/train_oasis2_colab.py',
    '--project-root', str(BACKEND_ROOT),
    '--runtime-root', str(RUNTIME_ROOT),
    '--bundle-root', str(OASIS2_BUNDLE_ROOT),
    '--run-name', RUN_NAME,
    '--epochs', '20',
    '--batch-size', '1',
    '--gradient-accumulation-steps', '1',
    '--num-workers', '0',
    '--image-size', '64', '64', '64',
    '--seed', '42',
    '--split-seed', '42',
    '--device', 'auto',
    '--stage-bundle-to-local',
]

print('RUNNING:', ' '.join(cmd))
subprocess.run(cmd, cwd=BACKEND_ROOT, check=True)


In [ ]:
from pathlib import Path
import json

blocked_summary_path = RUNTIME_ROOT / 'outputs' / 'reports' / 'onboarding' / 'oasis2_colab_bundle_summary.json'
trained_summary_path = RUNTIME_ROOT / 'outputs' / 'runs' / 'oasis2' / RUN_NAME / 'reports' / 'colab_run_summary.json'

print('blocked_summary_path =', blocked_summary_path)
print('trained_summary_path =', trained_summary_path)

summary_path = trained_summary_path if trained_summary_path.exists() else blocked_summary_path
if not summary_path.exists():
    raise FileNotFoundError(f'Expected an OASIS-2 summary JSON, but none was found: {summary_path}')

summary = json.loads(summary_path.read_text(encoding='utf-8'))
print(json.dumps(summary, indent=2))

if not summary.get('training_ready', False):
    print('Training is still blocked. Inspect training_readiness_json_path and any official_demographics_import_* paths in the summary for the remaining readiness gate.')
